# Notebook 02: BERT
Fine-tunes distilbert for multilabel emotion classification across fractions and seeds.


In [ ]:
# install dependencies and set up colab drive paths
!pip install -q -U "transformers<5" accelerate datasets

try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/cs4120_project/"
except Exception:
    SAVE_DIR = None

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.metrics import f1_score, classification_report
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

repo_candidates = [Path.cwd(), Path.cwd().parent]
repo_root = None
for candidate in repo_candidates:
    if (candidate / "src").exists() and (candidate / "data").exists():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError("Could not locate repo root with src/ and data/ directories")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

DATA_DIR = repo_root / "data"
RESULTS_DIR = repo_root / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Repo root:", repo_root)
print("Data dir:", DATA_DIR)
print("Results dir:", RESULTS_DIR)

Mounted at /content/drive
PyTorch: 2.10.0+cu128
GPU available: True
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
# force remount drive when needed in fresh runtimes
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# load splits and prepare multilabel training tensors
import ast
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
import ast, json, time
from pathlib import Path
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import Dataset
from transformers import (
   AutoTokenizer,
   AutoModelForSequenceClassification,
   TrainingArguments,
   Trainer,
   EarlyStoppingCallback,
   set_seed,
)


from pathlib import Path
# input/output paths for preprocessed data and experiment results
data_directory = Path("/content/drive/MyDrive/cs4120_project/data")
results_directory = Path("/content/drive/MyDrive/cs4120_project/results")
results_directory.mkdir(parents=True, exist_ok=True)
train_df = pd.read_csv(data_directory / "train.csv")
val_df = pd.read_csv(data_directory / "val.csv")
test_df = pd.read_csv(data_directory / "test.csv")
fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 1.0]
seeds = [42, 7, 21]

# convert serialized label strings to python lists
def parse_labels(x):
   if isinstance(x, list):
       return x
   try:
       return ast.literal_eval(x)
   except:
       return []


for df in [train_df, val_df, test_df]:
   df["labels"] = df["labels"].apply(parse_labels)


text_col = "text_clean_transformer"
for df in [train_df, val_df, test_df]:
    df["text"] = df[text_col].astype(str)


all_labels = set()
for labels in train_df["labels"]:
   all_labels.update(labels)


label_classes = sorted(list(all_labels))  
num_labels = len(label_classes)
label_names = label_classes



mlb = MultiLabelBinarizer(classes=label_classes)
mlb.fit([label_classes])


for df in [train_df, val_df, test_df]:
   df["labels_binary"] = list(mlb.transform(df["labels"]))



In [ ]:
# define dataset, metrics, and single-run distilbert training function
import sys
from pathlib import Path

exec(open("/content/drive/MyDrive/cs4120_project/src/evaluate.py").read())

#Dataset Class
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
from src.evaluate import evaluate_run

class EmotionDataset(Dataset):
   def __init__(self, df):
       texts = df["text"].astype(str).tolist()
       self.encodings = tokenizer(texts, truncation=True, padding="max_length", max_length=128, return_tensors="pt")
       labels_array = np.stack(df["labels_binary"].values)
       self.labels = torch.tensor(labels_array, dtype=torch.float32)


   def __len__(self):
       return len(self.labels)


   def __getitem__(self, idx):
       return {"input_ids": self.encodings["input_ids"][idx],
           "attention_mask": self.encodings["attention_mask"][idx],
           "labels": self.labels[idx]}



#get f1s
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    probs = torch.sigmoid(torch.tensor(logits)).numpy()

    preds = (probs >= 0.5).astype(int)
    
    f1_micro = f1_score(labels.astype(int), preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels.astype(int), preds, average="macro", zero_division=0)
    
    return {"f1_micro": f1_micro, "f1_macro": f1_macro}


# run one distilbert experiment for a single fraction/seed
def run(train_df, val_df, test_df, lr, bs, ep, wd, seed, data_fraction):
def run(train_df, val_df, test_df, lr, bs, ep, wd, seed, data_fraction):
   set_seed(seed)
   t0 = time.time()
  
   train_ds = EmotionDataset(train_df)
   val_ds = EmotionDataset(val_df)
   test_ds = EmotionDataset(test_df)
  
   model = AutoModelForSequenceClassification.from_pretrained(
       "distilbert-base-uncased", num_labels=num_labels, problem_type="multi_label_classification"
   )
  
   args = TrainingArguments(
       num_train_epochs=ep,
       per_device_train_batch_size=bs,
       per_device_eval_batch_size=bs*2,
       learning_rate=lr,
       weight_decay=wd,
       evaluation_strategy="epoch",
       save_strategy="no",
       logging_strategy="no",
       seed=seed,
       fp16=torch.cuda.is_available(),
   )
  
   trainer = Trainer(
       model=model,
       args=args,
       train_dataset=train_ds,
       eval_dataset=val_ds,
       compute_metrics=compute_metrics,
   )
  
   trainer.train()
  
   test_pred = trainer.predict(test_ds)
   logits_np = test_pred.predictions
   y_true = test_pred.label_ids
   y_pred = (torch.sigmoid(torch.tensor(logits_np)) >= 0.5).int().numpy()


   ev = evaluate_run(
       method="distilbert",
       data_fraction=data_fraction,
       seed=seed,
       label_names=label_names,
       y_true=y_true,
       y_pred=y_pred,
   )
  
   return {
       "frac": data_fraction,
       "seed": seed,
       "y_true": y_true,
       "y_pred": y_pred,
       "ev": ev,
       "elapsed_min": (time.time() - t0) / 60,
       "train_rows": len(train_df),
       "learning_rate": lr,
       "batch_size": bs,
       "num_epochs": ep,
       "weight_decay": wd,
   }

In [ ]:

# execute full fraction x seed sweep
fraction_results = []
fraction_results = []
for frac in fractions:
   for seed in seeds:
       df_frac = train_df.sample(frac=frac, random_state=seed).reset_index(drop=True)
       r = run(df_frac, val_df, test_df,lr=2e-5, bs=32, ep=1, wd=0.01, seed=seed, data_fraction=frac)
       fraction_results.append(r)
fraction_df = pd.DataFrame(fraction_results)  




overall = []
for idx, row in fraction_df.iterrows():
    eval_data = row["ev"]["overall"]
    overall.append({
        'method': 'distilbert',
        'data_fraction': float(row['frac']),
        'seed': int(row['seed']),
        'accuracy': float(eval_data.get('accuracy', 0)),
        'macro_f1': float(eval_data.get('macro_f1', 0)),
        'micro_f1': float(eval_data.get('micro_f1', 0)),
        'hamming_loss': float(eval_data.get('hamming_loss', 0)),
        'train_rows': float(row.get('train_rows', 0)),	
        'learning_rate': float(row.get('learning_rate', 0)),	
        'batch_size': float(row.get('batch_size', 0)),	
        'num_epochs': float(row.get('num_epochs', 0)),	
        'weight_decay':float(row.get('weight_decay', 0)),	
        'elapsed_min':float(row.get('elapsed_min', 0)),
    })
overall_df = pd.DataFrame(overall)
overall_df.to_csv(RESULTS_DIR / "distilbert_overall.csv", index=False)



per_class = []
for index, row in fraction_df.iterrows():
    per_class_data = row["ev"]["per_class"].copy()
    per_class_data['method'] = 'distilbert'
    per_class_data['data_fraction'] = float(row['frac'])
    per_class_data['seed'] = int(row['seed'])
    cols = ['method', 'data_fraction', 'seed', 'emotion', 'precision', 'recall', 'f1', 'support', 'tp', 'fp', 'fn', 'tn']
    per_class_data = per_class_data[cols]
    per_class.append(per_class_data)

per_class_df = pd.concat(per_class, ignore_index=True)
per_class_df.to_csv(RESULTS_DIR / "distilbert_per_class.csv", index=False)




results_df = per_class_df[['method', 'data_fraction', 'seed', 'emotion', 'f1', 'precision', 'recall']].copy()
results_df.to_csv(RESULTS_DIR / "distilbert_results.csv", index=False)


